# 파이프 라인 사용법
my_pipeline = pipeline("the_task_I_want_to_do")  이게 파이프라인 API
result = my_pipeline(my_input)

# 훈련과 추론의 차이
데이터 과학 모델을 처음 사용할 때 가장 먼저 해야 할 일은 많은 데이터와 다양한 예제를 제공하여 원하는 작업을 잘 수행할 수 있도록 훈련 시키는 것입니다. 

GPT의 'P'는 **사전 학습(Pretrained)**을 의미합니다. 이는 OpenAI가 다음 토큰을 더 잘 예측하도록 만들기 위해, 인터넷의 방대한 데이터를 활용해 무려 1억 달러에 달하는 비용을 투자했다는 뜻입니다. 이 훈련 과정은 엄청나게 고되고 힘든 작업입니다.


이 과정이 끝나면 사전 학습된 모델을 갖게 됩니다. 그리고 이 시점부터 우리가 하는 활동을 **추론(Inference)**이라고 부릅니다.
추론이란, 훈련 중에는 한 번도 본 적 없는 새로운 입력이 들어왔을 때 즉 학습 데이터에는 없던 무언가가 주어졌 때 "다음에 어떤 일이 일어날지 예측해봐", "이 새로운 입력에 대해 작업을 수행해봐" 라고 요청하는 것을 말합니다.

다시 말해, 학습이 끝난 모델을 실제로 실행하는 것을 추론이라고 합니다. 거창한 용어처럼 들리지만 결국 "모델을 실행해서 오늘은 추론을 해보겠다"는 의미일 뿐입니다.

파이프 라인 API는 바로 이 추론을 위한 것입니다.

In [ ]:
!pip install -q --upgrade datasets==3.6.0 transformers==4.57.6 huggingface_hub diffusers

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.6.0+cu124 requires torch==2.6.0+cu124, but you have torch 2.4.1 which is incompatible.
trl 1.9.2 requires datasets>=4.7.0, but you have datasets 3.6.0 which is incompatible.


: 

모델을 별도로 지정하지 않으면 Hugging Face가 해당 작업의 기본 모델을 자동으로 선택합니다. 
T4와 같은 NVIDIA GPU를 사용하려면 장치(device)를 "cuda"로 지정하고, Mac에서는 "mps"로 지정하십시오.
my_pipeline = pipeline(task, model=xx, device=xx)


In [ ]:
import torch
from google.colab import userdata
from huggingface_hub import login
from transformers import pipeline
from diffusers import DiffusionPipeline
from datasets import load_dataset
import soundfile as sf
from IPython.display import Audio

In [ ]:
hf_token = userdata.get('HF_TOKEN')
if hf_token and hf_token.startswith("hf_"):
  print("HF key looks good so far")
else:
  print("HF key is not set - please click the key in the left sidebar")
login(hf_token, add_to_git_credential=True)

In [ ]:
my_simple_sentiment_analyzer = pipeline("sentiment-analysis", device="cuda")
result = my_simple_sentiment_analyzer("LLM을 완벽하게 마스터하는 길로 들어서게 되어 정말 설렙니다!")
print(result)

# [{'label': 'POSITIVE', 'score': 0.9262627959251404}]

In [ ]:
result = my_simple_sentiment_analyzer("LLM을 마스터하는 과정에 있다는 사실에 더 설레야겠어요!")
print(result)

# [{'label': 'POSITIVE', 'score': 0.8392764329910278}]

In [ ]:
better_sentiment = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment", device="cuda")
result = better_sentiment("I should be more excited to be on the way to LLM mastery!!")
print(result)

In [ ]:
# 개체명 인식 (Named Entity Recognition)
ner = pipeline("ner", device="cuda")
result = ner("AI Engineers are learning about the amazing pipelines from HuggingFace in Google Colab from Ed Donner")
for entity in result:
  print(entity)



# {'entity': 'I-ORG', 'score': np.float32(0.999476), 'index': 1, 'word': 'AI', 'start': 0, 'end': 2}
# {'entity': 'I-ORG', 'score': np.float32(0.9962089), 'index': 2, 'word': 'Engineers', 'start': 3, 'end': 12}
# {'entity': 'I-ORG', 'score': np.float32(0.8205678), 'index': 11, 'word': 'Hu', 'start': 59, 'end': 61}
# {'entity': 'I-ORG', 'score': np.float32(0.651493), 'index': 12, 'word': '##gging', 'start': 61, 'end': 66}
# {'entity': 'I-ORG', 'score': np.float32(0.960993), 'index': 13, 'word': '##F', 'start': 66, 'end': 67}
# {'entity': 'I-ORG', 'score': np.float32(0.92517126), 'index': 14, 'word': '##ace', 'start': 67, 'end': 70}
# {'entity': 'I-MISC', 'score': np.float32(0.8882749), 'index': 16, 'word': 'Google', 'start': 74, 'end': 80}
# {'entity': 'I-MISC', 'score': np.float32(0.67307675), 'index': 17, 'word': 'Cola', 'start': 81, 'end': 85}
# {'entity': 'I-PER', 'score': np.float32(0.9989441), 'index': 20, 'word': 'Ed', 'start': 92, 'end': 94}
# {'entity': 'I-PER', 'score': np.float32(0.99872154), 'index': 21, 'word': 'Don', 'start': 95, 'end': 98}
# {'entity': 'I-PER', 'score': np.float32(0.96185666), 'index': 22, 'word': '##ner', 'start': 98, 'end': 101}

In [ ]:
# 문맥을 활용한 질의응답


question="허깅페이스 파이프라인이 뭔가요?"
context="파이프라인(Pipelines)은 일반적인 작업에 대해 LLM 추론을 수행하기 위한 상위 수준의 API입니다."

question_answerer = pipeline("question-answering", device="cuda")
result = question_answerer(question=question, context=context)
print(result)

In [ ]:
# Text Summarization

summarizer = pipeline("summarization", device="cuda")
text = """
Hugging Face의 transformers 라이브러리는 자연어 처리(NLP) 분야에서 매우 다재다능하고 강력한 도구입니다.
이 라이브러리를 사용하면 텍스트 분류, 개체명 인식, 질의응답 등 다양한 작업을 수행할 수 있습니다.
또한 오픈 소스 데이터 과학 커뮤니티에서 널리 사용되는 매우 인기 있는 라이브러리이기도 합니다.
데이터 과학자들에게 트랜스포머(Transformer) 모델을 다루는 생산적이고 편리한 방법을 제공함으로써, 해당 분야로의 진입 장벽을 낮춰줍니다.
"""

summary = summarizer(text, max_length=50, min_length=25, do_sample=False)
print(summary[0]['summary_text'])



In [ ]:
# Translation

translator = pipeline("translation_en_to_fr", device="cuda")
result = translator("The Data Scientists were truly amazed by the power and simplicity of the HuggingFace pipeline API.")
print(result[0]['translation_text'])

In [ ]:
# Another translation, showing a model being specified
# All translation models are here: https://huggingface.co/models?pipeline_tag=translation&sort=trending

translator = pipeline("translation_en_to_es", model="Helsinki-NLP/opus-mt-en-es", device="cuda")
result = translator("The Data Scientists were truly amazed by the power and simplicity of the HuggingFace pipeline API.")
print(result[0]['translation_text'])

In [ ]:
# Classification(분류)
# Zero-shot 분류기는 어떤 예시도 제공해 주지 않겠다는 뜻이다.

classifier = pipeline("zero-shot-classification", device="cuda")
result = classifier("Hugging Face's Transformers library is amazing!", candidate_labels=["technology", "sports", "politics"])
print(result)

# {'sequence': "Hugging Face's Transformers library is amazing!", 'labels': ['technology', 'sports', 'politics'], 'scores': [0.9493839740753174, 0.03225007280707359, 0.018365919589996338]}

In [ ]:
# Text Generation

generator = pipeline("text-generation", device="cuda")
result = generator("If there's one thing I want you to remember about using HuggingFace pipelines, it's")
print(result[0]['generated_text'])



In [ ]:
# Image Generation - remember this?! Now you know what's going on
# 파이프라인은 트랜스포머뿐만 아니라 확산 모델(diffusion models)에도 사용할 수 있습니다.

from IPython.display import display
from diffusers import AutoPipelineForText2Image
import torch

pipe = AutoPipelineForText2Image.from_pretrained("stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16")
pipe.to("cuda")
prompt = "A class of students learning AI engineering in a vibrant pop-art style"
image = pipe(prompt=prompt, num_inference_steps=4, guidance_scale=0.0).images[0]
display(image)

In [ ]:
# Audio Generation

from transformers import pipeline
from datasets import load_dataset
import soundfile as sf
import torch
from IPython.display import Audio

synthesiser = pipeline("text-to-speech", "microsoft/speecht5_tts", device='cuda')
embeddings_dataset = load_dataset("matthijs/cmu-arctic-xvectors", split="validation", trust_remote_code=True)
speaker_embedding = torch.tensor(embeddings_dataset[7306]["xvector"]).unsqueeze(0)
speech = synthesiser("Hi to an artificial intelligence engineer, on the way to mastery!", forward_params={"speaker_embeddings": speaker_embedding})

Audio(speech["audio"], rate=speech["sampling_rate"])